In [36]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
!mkdir /content/drive/MyDrive/potato_data

mkdir: cannot create directory ‘/content/drive/MyDrive/potato_data’: File exists


In [ ]:
!unzip /content/drive/MyDrive/potato_dataset.zip -d /content/drive/MyDrive/potato_data

Archive:  /content/drive/MyDrive/potato_dataset.zip
replace /content/drive/MyDrive/potato_data/archive/PLD_3_Classes_256/Testing/Early_Blight/034959c1-f1e8-4a79-a6d5-3c1d14efa2f3___RS_Early.B 7136.JPG? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [39]:
!ls /content/drive/MyDrive/potato_data


archive


In [40]:
!ls /content/drive/MyDrive/potato_data/Training

ls: cannot access '/content/drive/MyDrive/potato_data/Training': No such file or directory


In [41]:
!ls /content/drive/MyDrive/potato_data

archive


In [42]:
!ls /content/drive/MyDrive/potato_data/archive

PLD_3_Classes_256


In [43]:
!ls /content/drive/MyDrive/potato_data/archive/PLD_3_Classes_256

Testing  Training  Validation


In [44]:
!ls /content/drive/MyDrive/potato_data/archive/Training

ls: cannot access '/content/drive/MyDrive/potato_data/archive/Training': No such file or directory


In [45]:
!ls /content/drive/MyDrive/potato_data/archive/PLD_3_Classes_256/Training


Early_Blight  Healthy  Late_Blight


In [46]:
train_dir = "/content/drive/MyDrive/potato_data/archive/PLD_3_Classes_256/Training"
val_dir   = "/content/drive/MyDrive/potato_data/archive/PLD_3_Classes_256/Validation"
test_dir  = "/content/drive/MyDrive/potato_data/archive/PLD_3_Classes_256/Testing"

In [47]:
!ls /content/drive/MyDrive/potato_data/archive/PLD_3_Classes_256

Testing  Training  Validation


In [48]:
!ls /content/drive/MyDrive/potato_data/archive/PLD_3_Classes_256/Training


Early_Blight  Healthy  Late_Blight


In [49]:
!ls /content/drive/MyDrive/potato_data/archive/PLD_3_Classes_256/Training/healthy

ls: cannot access '/content/drive/MyDrive/potato_data/archive/PLD_3_Classes_256/Training/healthy': No such file or directory


In [50]:
!ls /content/drive/MyDrive/potato_data/archive/PLD_3_Classes_256/Training/Healthy

Healthy_100.jpg  Healthy_436.jpg  Healthy_771.jpg
Healthy_101.jpg  Healthy_437.jpg  Healthy_772.jpg
Healthy_102.jpg  Healthy_438.jpg  Healthy_773.jpg
Healthy_103.jpg  Healthy_439.jpg  Healthy_774.jpg
Healthy_104.jpg  Healthy_43.jpg   Healthy_775.jpg
Healthy_105.jpg  Healthy_440.jpg  Healthy_776.jpg
Healthy_106.jpg  Healthy_441.jpg  Healthy_777.jpg
Healthy_107.jpg  Healthy_442.jpg  Healthy_778.jpg
Healthy_108.jpg  Healthy_443.jpg  Healthy_779.jpg
Healthy_109.jpg  Healthy_444.jpg  Healthy_77.jpg
Healthy_10.jpg	 Healthy_445.jpg  Healthy_780.jpg
Healthy_110.jpg  Healthy_446.jpg  Healthy_781.jpg
Healthy_111.jpg  Healthy_447.jpg  Healthy_782.jpg
Healthy_112.jpg  Healthy_448.jpg  Healthy_783.jpg
Healthy_113.jpg  Healthy_449.jpg  Healthy_784.jpg
Healthy_114.jpg  Healthy_44.jpg   Healthy_785.jpg
Healthy_115.jpg  Healthy_450.jpg  Healthy_786.jpg
Healthy_116.jpg  Healthy_451.jpg  Healthy_787.jpg
Healthy_117.jpg  Healthy_452.jpg  Healthy_788.jpg
Healthy_118.jpg  Healthy_453.jpg  Healthy_789.jpg
He

In [51]:
import os
import cv2
import numpy as np


In [52]:
IMG_SIZE = 224

In [53]:
def load_and_preprocess(folder_path):
    images = []
    labels = []

    class_names = ['Healthy', 'Early_Blight', 'Late_Blight']

    for label, class_name in enumerate(class_names):
        class_folder = os.path.join(folder_path, class_name)

        for img_name in os.listdir(class_folder):
            img_path = os.path.join(class_folder, img_name)

            img = cv2.imread(img_path)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = img / 255.0   # normalization

            images.append(img)
            labels.append(label)

    return np.array(images), np.array(labels)


In [ ]:
X_train, y_train = load_and_preprocess(train_dir)
X_val, y_val     = load_and_preprocess(val_dir)
X_test, y_test   = load_and_preprocess(test_dir)


In [ ]:
print("Train shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_val.shape, y_val.shape)
print("Test shape:", X_test.shape, y_test.shape)

print("Pixel value range:", X_train.min(), "to", X_train.max())


In [ ]:
train_aug = ImageDataGenerator(
    rotation_range=20,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2]
)

val_aug = ImageDataGenerator()   # NO augmentation
test_aug = ImageDataGenerator()  # NO augmentation


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
train_aug = ImageDataGenerator(
    rotation_range=20,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2]
)

val_aug = ImageDataGenerator()   # NO augmentation
test_aug = ImageDataGenerator()  # NO augmentation


In [ ]:
train_generator = train_aug.flow(X_train, y_train, batch_size=32, shuffle=True)
val_generator   = val_aug.flow(X_val, y_val, batch_size=32, shuffle=False)
test_generator  = test_aug.flow(X_test, y_test, batch_size=32, shuffle=False)


In [ ]:
batch_x, batch_y = next(train_generator)
print("Batch shape:", batch_x.shape, batch_y.shape)
print("Pixel range:", batch_x.min(), "to", batch_x.max())


In [ ]:
import matplotlib.pyplot as plt


In [ ]:
plt.figure(figsize=(10,5))

for i in range(5):
    plt.subplot(2,3,i+1)
    plt.imshow(X_train[i])
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")

plt.tight_layout()
plt.show()
